In [1]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
!unzip /content/drive/MyDrive/content/train.zip -d /content/dataset

Archive:  /content/drive/MyDrive/content/train.zip
   creating: /content/dataset/train/
  inflating: /content/dataset/__MACOSX/._train  
  inflating: /content/dataset/train/.DS_Store  
  inflating: /content/dataset/__MACOSX/train/._.DS_Store  
  inflating: /content/dataset/train/data.yaml  
  inflating: /content/dataset/__MACOSX/train/._data.yaml  
   creating: /content/dataset/train/train/
  inflating: /content/dataset/__MACOSX/train/._train  
  inflating: /content/dataset/train/train/.DS_Store  
  inflating: /content/dataset/__MACOSX/train/train/._.DS_Store  
   creating: /content/dataset/train/train/images/
  inflating: /content/dataset/__MACOSX/train/train/._images  
   creating: /content/dataset/train/train/labels/
  inflating: /content/dataset/__MACOSX/train/train/._labels  
  inflating: /content/dataset/train/train/images/20260201_070904_015975_jpg.rf.7b8d3ad351de4a5a242da201da5503a3.jpg  
  inflating: /content/dataset/__MACOSX/train/train/images/._20260201_070904_015975_jpg.rf.

In [ ]:
from google.colab import files
files.upload()  # select all 3 files

In [ ]:
%%writefile train.py
from ultralytics import YOLO

def train_model():
    model = YOLO('yolov8n.pt')

    model.train(
        data='data.yaml',
        epochs=20,
        imgsz=640,
        batch=16,
        name='damage_detection'
    )

if __name__ == "__main__":
    train_model()

In [ ]:
%%writefile data.yaml
train: /content/dataset/train/train/images
val: /content/dataset/train/train/images

nc: 1
names: ["damage"]

In [ ]:
%%writefile pipeline.py
from ultralytics import YOLO
import cv2
import json
import os

MODEL_PATH = '/content/runs/detect/damage_detection3/weights/best.pt'
INPUT_FOLDER = '/content/dataset/train/train/images'
OUTPUT_FOLDER = '/content/outputs'

model = YOLO(MODEL_PATH)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

for img_name in os.listdir(INPUT_FOLDER):
    img_path = os.path.join(INPUT_FOLDER, img_name)
    results = model(img_path)[0]

    detections = {}
    img = cv2.imread(img_path)

    if results.boxes is not None:
        for i, box in enumerate(results.boxes.xyxy):
            x_min, y_min, x_max, y_max = map(int, box.tolist())

            detections[str(i+1)] = {
                "bbox_coordinates": [x_min, y_min, x_max, y_max]
            }

            cv2.rectangle(img, (x_min, y_min), (x_max, y_max), (0,255,0), 2)

    cv2.imwrite(os.path.join(OUTPUT_FOLDER, img_name), img)

    json_path = os.path.join(OUTPUT_FOLDER, img_name.split('.')[0] + '.json')
    with open(json_path, 'w') as f:
        json.dump(detections, f, indent=4)

    print(f"Processed: {img_name}")

In [ ]:
!ls

In [ ]:
!pip install ultralytics opencv-python

In [ ]:
%%writefile data.yaml
train: /content/dataset/train/train/images
val: /content/dataset/train/train/images

nc: 1
names: ["damage"]

In [ ]:
!pip install ultralytics

In [ ]:
!ls

In [ ]:
!python train.py --data_dir /content/train --epochs 50 --batch 16 --output_dir /content/runs

In [ ]:
!ls

In [ ]:
!find /content/dataset -type d

In [ ]:
!ls runs/detect/damage_detection3/weights/

In [ ]:
!python pipeline.py \
  --image_dir /content/train/images \
  --output_dir /content/outputs \
  --weights /content/runs/belt_damage/weights/best.pt

In [ ]:
import shutil
shutil.make_archive('/content/outputs', 'zip', '/content/outputs')

from google.colab import files
files.download('/content/outputs.zip')
files.download('/content/runs/detect/damage_detection3/weights/best.pt')